# Volume of Mixing vs Pressure

Computes ΔV_mix(P*) for 11 pressures P* = 1.0 – 2.0.

## Definition

$$\Delta V_{\rm mix} = V_{\rm mix} - V_{\rm pol} - V_{\rm sol}$$

All three are **NPT-equilibrated box volumes** at the same P*, read from
`box_dimensions_*.dat`:

- **V_mix** — the isolated gel (all polymer + all solvent, `isolated_*.data`)
  re-equilibrated fully periodic under NPT (`polymer_pure` engine, `1.0_1.0`).
- **V_pol** — pure polymer (`polymer_pure`, `*_polymer_only`, `1.0_0.0`).
- **V_sol** — pure solvent (`solvent_pure`, `*_solvent_only`, `1.0_0.0`).

No trimming and no scaling: the pure boxes hold exactly the same atoms as the
mix, so $N_{\rm pol}$ and $N_{\rm sol}$ match across all three runs (scale = 1).

## Scientific basis for this pipeline (2026-06 redesign)

- **The dominant error was an inconsistent pair potential, not geometry.**
  `polymer_pure` previously equilibrated with attractive LJ (`lj/cut 2.5`) while
  the gel is WCA (`lj/cut 1.122`). Switching it to WCA cut
  ΔV_mix/(V_sol+V_pol) ~10× (≈0.20 → ≈0.02). All three boxes now use identical
  WCA (`lj/cut 1.122`, ε = 1).
- **V_mix must be the same kind of volume as the references.** It was a
  geometric bounding box (`isolate_gel.py` extent + clearance); the references
  were NPT volumes. Subtracting different volume types is not a valid ΔV_mix.
  V_mix is now an NPT box volume measured identically to V_pol and V_sol.
- **Clearance was a vertical offset, not the trend** (clearance-sensitivity
  sweep): deflating the isolate clearance 0.2 → 0 shifted every point by a
  near-constant ≈0.018 in ΔV_mix/(V_sol+V_pol) (0.0180 at P*=1.0, 0.0184 at
  P*=2.0) with unchanged slope (~0.0088 → 0.0084 per unit P*). Removing it
  cleans the absolute baseline but cannot explain the pressure dependence.
- **The remaining pressure dependence is physical packing frustration.** Bonded
  beads sit at the FENE equilibrium ≈0.97σ; nonbonded beads contact at
  2^(1/6)σ ≈ 1.122σ. This ~13 % size/spacing asymmetry produces excess volume
  that grows under compression. Matching the two lengths would destroy the
  Kremer–Grest no-chain-crossing property, so it is retained. Expect a small
  (~1–3 %), rising ΔV_mix as a model property, not a bug.
- **Trimming/scaling removed.** All polymer and all solvent seed both the pure
  boxes and the mix, eliminating the per-species percentile-trim
  density-extrapolation assumption (scale factors are identically 1).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import glob
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

plt.rcParams.update({
    'font.family':        'CMU Serif',
    'mathtext.fontset':   'cm',
    'mathtext.rm':        'CMU Serif',
    'font.size':          20,
    'axes.titlesize':     22,
    'axes.labelsize':     25,
    'xtick.labelsize':    23,
    'ytick.labelsize':    23,
    'legend.fontsize':    23,
    'figure.titlesize':   22,
    'axes.unicode_minus': False,
    'figure.dpi':         120,
})

# --- Configuration ---
BASE_DATANAME  = "slab_support_5beads_tall_rho04"
INTERACTION    = "1.0_1.0"
PURE_INTER     = "1.0_0.0"
SLAB_STEPS     = 600000
PURE_STEPS     = 100000

DATA_DIR = Path("../../flow_data_local/volmix_sweep")

PRESSURES = [round(1.0 + i * 0.1, 1) for i in range(11)]
print(f"Pressures: {PRESSURES}")
print(f"Data root: {DATA_DIR.resolve()}")


In [ ]:
# === Sync volume data from Expanse ===
import paramiko, getpass, stat
from pathlib import Path

EXPANSE_HOST = "login.expanse.sdsc.edu"
EXPANSE_USER = "dpollard"
STAGE_DIR    = "/home/dpollard/Documents/lammps_runs/volmix_sweep/volmix_stage"

# Stage the THREE NPT box_dimensions files (mix, pure polymer, pure solvent) for
# each pressure, located by exact filename anywhere under the sweep tree so that
# reusing the polymer_pure folder for the mixed run can't cause a glob collision.
stage_script = (
    "SWEEP=~/Documents/lammps_runs/volmix_sweep\n"
    "DATA=~/Documents/lammps_data\n"
    "STAGE=${SWEEP}/volmix_stage\n"
    "BASE=slab_support_5beads_tall_rho04\n"
    "mkdir -p \"$STAGE\"\n"
    "for P in 1.0 1.1 1.2 1.3 1.4 1.5 1.6 1.7 1.8 1.9 2.0; do\n"
    "  mkdir -p \"$STAGE/p${P}\"\n"
    "  ISTEM=\"isolated_${BASE}_pstar${P}_1.0_1.0_600000\"\n"
    "  MIX_F=\"box_dimensions_${ISTEM}_1.0_1.0_100000.dat\"\n"
    "  POL_F=\"box_dimensions_${ISTEM}_polymer_only_1.0_0.0_100000.dat\"\n"
    "  SOL_F=\"box_dimensions_${ISTEM}_solvent_only_1.0_0.0_100000.dat\"\n"
    "  for F in \"$MIX_F\" \"$POL_F\" \"$SOL_F\"; do\n"
    "    SRC=$(find \"$SWEEP\" -name \"$F\" -not -path '*/volmix_stage/*' -printf '%T@ %p\\n' 2>/dev/null | sort -rn | head -1 | cut -d' ' -f2-)\n"
    "    [ -n \"$SRC\" ] && cp \"$SRC\" \"$STAGE/p${P}/\" 2>/dev/null || true\n"
    "  done\n"
    "  MANIFEST=\"$DATA/input_data/${ISTEM}_volmix_manifest.json\"\n"
    "  [ -f \"$MANIFEST\" ] && cp \"$MANIFEST\" \"$STAGE/p${P}/\" 2>/dev/null || true\n"
    "  cnt=$(ls \"$STAGE/p${P}/\" 2>/dev/null | wc -l)\n"
    "  echo \"  P=${P}: $cnt files staged\"\n"
    "done\n"
)

password = getpass.getpass(f"Expanse password for {EXPANSE_USER}: ")
totp     = getpass.getpass("TOTP / verification code: ")

def auth_handler(title, instructions, prompt_list):
    responses = []
    for prompt, echo in prompt_list:
        if "password" in prompt.strip().lower():
            responses.append(password)
        else:
            responses.append(totp)
    return responses

print("Connecting to Expanse...")
transport = paramiko.Transport((EXPANSE_HOST, 22))
transport.connect()
transport.auth_interactive(EXPANSE_USER, auth_handler)

ssh = paramiko.SSHClient()
ssh._transport = transport

print("Step 1 — staging files on Expanse...")
_, stdout, stderr = ssh.exec_command("bash -s", get_pty=False)
stdout.channel.sendall(stage_script.encode())
stdout.channel.shutdown_write()
print(stdout.read().decode())

print("Step 2 — downloading via SFTP (skips files already present)...")
sftp = ssh.open_sftp()
DATA_DIR.mkdir(parents=True, exist_ok=True)

def sftp_download_dir(sftp, remote_dir, local_dir):
    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)
    for entry in sftp.listdir_attr(remote_dir):
        remote_path = f"{remote_dir}/{entry.filename}"
        local_path  = local_dir / entry.filename
        if stat.S_ISDIR(entry.st_mode):
            sftp_download_dir(sftp, remote_path, local_path)
        else:
            if local_path.exists() and local_path.stat().st_mtime >= entry.st_mtime:
                continue
            sftp.get(remote_path, str(local_path))

sftp_download_dir(sftp, STAGE_DIR, DATA_DIR)
sftp.close()
ssh.close()
print("Sync complete.")


## Volume parsing functions

In [ ]:
def avg_box_volume(path, skip_frac=0.5):
    """
    Parse box_dimensions_*.dat (columns: step lx ly lz).
    Returns time-averaged volume = mean(lx*ly*lz) over the last (1-skip_frac) fraction.
    """
    data = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) >= 4:
                try:
                    step, lx, ly, lz = float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3])
                    data.append(lx * ly * lz)
                except ValueError:
                    continue
    if not data:
        raise ValueError(f"No data found in {path}")
    data = np.array(data)
    n_skip = int(len(data) * skip_frac)
    return np.mean(data[n_skip:])


def avg_pure_volume(path, skip_frac=0.0):
    """
    Parse vol_pure_*.dat (columns: step press_mean vol_mean rho_mean, block averages).
    Returns mean of vol_mean column over all blocks (skip_frac=0 since runs are short).
    """
    data = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) >= 3:
                try:
                    vol = float(parts[2])  # vol_mean column
                    data.append(vol)
                except ValueError:
                    continue
    if not data:
        raise ValueError(f"No data found in {path}")
    data = np.array(data)
    n_skip = int(len(data) * skip_frac)
    return np.mean(data[n_skip:])


def load_manifest(path):
    """Load the volmix manifest JSON written by split_gel.py."""
    with open(path) as f:
        return json.load(f)


## Load volume data for each pressure

Uses the sweep manifest files written by pressure_sweep.sh to locate each run directory.

In [ ]:
import json

results = []
missing = []

for P in PRESSURES:
    pstr = f"{P:.1f}"
    dataname          = f"{BASE_DATANAME}_pstar{pstr}"
    isolated_dataname = f"isolated_{dataname}_{INTERACTION}_{SLAB_STEPS}"
    sol_dataname      = f"{isolated_dataname}_solvent_only"
    pol_dataname      = f"{isolated_dataname}_polymer_only"

    p_dir = DATA_DIR / f"p{pstr}"

    # All three are NPT box_dimensions now (same estimator); no trim -> scale=1.
    mix_vol_file = p_dir / f"box_dimensions_{isolated_dataname}_{INTERACTION}_{PURE_STEPS}.dat"
    sol_vol_file = p_dir / f"box_dimensions_{sol_dataname}_{PURE_INTER}_{PURE_STEPS}.dat"
    pol_vol_file = p_dir / f"box_dimensions_{pol_dataname}_{PURE_INTER}_{PURE_STEPS}.dat"

    if not all(f.exists() for f in [mix_vol_file, sol_vol_file, pol_vol_file]):
        missing.append(pstr)
        for label, fpath in [("mix", mix_vol_file), ("solvent", sol_vol_file), ("polymer", pol_vol_file)]:
            if not fpath.exists():
                print(f"[SKIP] P*={pstr}: missing {label} -> {fpath}")
        continue

    try:
        V_mix    = avg_box_volume(mix_vol_file, skip_frac=0.5)   # mixed gel NPT
        V_pol_eq = avg_box_volume(pol_vol_file, skip_frac=0.5)   # pure polymer NPT
        V_sol_eq = avg_box_volume(sol_vol_file, skip_frac=0.5)   # pure solvent NPT

        # Identical atom counts across mix and pure runs (no trim) -> scale = 1.
        dV = V_mix - V_pol_eq - V_sol_eq

        # Optional sanity check: manifest scale factors should be 1.0.
        manifest_file = p_dir / f"{isolated_dataname}_volmix_manifest.json"
        if manifest_file.exists():
            mf = load_manifest(manifest_file)
            if (abs(mf.get('scale_pol', 1.0) - 1.0) > 1e-6 or
                    abs(mf.get('scale_sol', 1.0) - 1.0) > 1e-6):
                print(f"[WARN] P*={pstr}: manifest scale != 1 "
                      f"(scale_pol={mf.get('scale_pol')}, scale_sol={mf.get('scale_sol')}) "
                      f"-- trimming may still be enabled in split_gel.py")

        results.append({
            "P": P, "V_mix": V_mix, "V_sol": V_sol_eq,
            "V_pol": V_pol_eq, "dV_mix": dV,
        })
        print(f"P*={pstr}:  V_mix={V_mix:.2f}  V_sol={V_sol_eq:.2f}  "
              f"V_pol={V_pol_eq:.2f}  dV={dV:+.3f}")
    except Exception as e:
        print(f"[ERROR] P*={pstr}: {e}")
        missing.append(pstr)

df = pd.DataFrame(results)
print(f"\nLoaded {len(df)}/{len(PRESSURES)} pressure points")
if missing:
    print(f"Missing: {missing}")


## Plot ΔV_mix vs P*

In [ ]:
if df.empty:
    print("No data to plot — run pressure_sweep.sh and wait for all jobs to complete.")
else:
    P      = df["P"].values
    V_mix  = df["V_mix"].values
    V_sol  = df["V_sol"].values
    V_pol  = df["V_pol"].values
    dV_mix = df["dV_mix"].values
    V_ref  = V_sol + V_pol        # isolated pure-component total

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- Left: ΔV_mix / V_ref  (fractional volume of mixing) ---
    ax = axes[0]
    ax.plot(P, dV_mix / V_ref, "o-", color="steelblue", lw=2, ms=7)
    ax.axhline(0, color="gray", lw=1, ls="--")
    ax.set_xlabel(r"$P^*$")
    ax.set_ylabel(r"$\Delta V_{\rm mix}\;/\;(V_{\rm sol}+V_{\rm pol})$")
    ax.set_title("Volume of Mixing")
    ax.grid(True, alpha=0.3)

    # --- Right: component volumes / V_mix  (volume fractions) ---
    ax2 = axes[1]
    ax2.plot(P, V_sol / V_mix * 100, "s--", label=r"$V_{\rm solvent}/V_{\rm mix}$", color="tomato",   lw=1.5, ms=5)
    ax2.plot(P, V_pol / V_mix * 100, "^--", label=r"$V_{\rm polymer}/V_{\rm mix}$", color="seagreen",  lw=1.5, ms=5)
    ax2.set_xlabel(r"$P^*$")
    ax2.set_ylabel(r"$V_i\;/\;V_{\rm mix}\;[\%]$")
    ax2.set_title("Volume Fractions (isolated gel)")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    PLOT_DIR = Path("../../flow_data_local/plots/volmix")
    PLOT_DIR.mkdir(parents=True, exist_ok=True)
    plt.savefig(PLOT_DIR / "volume_of_mixing.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {PLOT_DIR / 'volume_of_mixing.png'}")


## Summary table

In [ ]:
if not df.empty:
    from IPython.display import display
    display_df = df.copy()
    display_df.columns = ["P*", "V_mix [σ³]", "V_solvent [σ³]", "V_polymer [σ³]", "ΔV_mix [σ³]"]
    display_df = display_df.round(3)
    display(display_df)


In [ ]:
# =============================================================================
# Clearance-sensitivity check for ΔV_mix  (standalone notebook cell)
# =============================================================================
# Diagnostic only — does NOT touch the pipeline.
#
# V_mix is currently the geometric bounding box that isolate_gel.py draws around
# the polymer: (polymer extent) + BOX_CLEARANCE on every face. That clearance is
# a *fixed* 0.2σ per face (0.4σ per axis). As the gel compresses with P*, the
# fixed margin becomes a larger FRACTION of a shrinking box, which biases
# ΔV_mix upward with pressure.
#
# This cell recomputes ΔV_mix while shrinking the clearance from the original
# 0.2 down to 0, using the SAME pure-phase references (V_pol, V_sol unchanged).
# If the upward slope mostly flattens as clearance → 0, the trend was the
# clearance artifact. If it persists, it's the real connectivity/packing effect
# and the full NPT-V_mix restructure is what's needed to confirm.
#
# Assumes the notebook has already defined (from the earlier cells):
#   PRESSURES, BASE_DATANAME, INTERACTION, SLAB_STEPS, DATA_DIR,
#   PURE_INTER, PURE_STEPS, load_manifest, avg_box_volume, np, plt
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ORIG_CLEARANCE = 0.2          # BOX_CLEARANCE used in isolate_gel.py (per face)
TEST_CLEARANCES = [0.2, 0.15, 0.10, 0.05, 0.0]   # per face; 0.2 == current pipeline


def _read_box_dims_from_data(path):
    """Return (Lx, Ly, Lz) from a LAMMPS data-file header, or None if not found."""
    lo_hi = {}
    try:
        with open(path) as f:
            for line in f:
                for axis in ("x", "y", "z"):
                    if f"{axis}lo {axis}hi" in line:
                        p = line.split()
                        lo_hi[axis] = (float(p[0]), float(p[1]))
                if len(lo_hi) == 3:
                    break
    except FileNotFoundError:
        return None
    if len(lo_hi) != 3:
        return None
    return tuple(hi - lo for lo, hi in (lo_hi["x"], lo_hi["y"], lo_hi["z"]))


def _find_isolated_dims(p_dir, isolated_dataname):
    """
    Try to get the per-axis box of the isolated mixed gel (exact deflation).
    Falls back to None -> caller uses the cube-root approximation.
    Searches a few plausible local locations for isolated_*.data.
    """
    candidates = [
        p_dir / f"{isolated_dataname}.data",
        p_dir / f"{isolated_dataname}.lammps",
        # add your own local path here if the isolated files live elsewhere:
        # Path.home() / "Documents/lammps_data/slab_with_support" / f"{isolated_dataname}.data",
    ]
    for c in candidates:
        dims = _read_box_dims_from_data(c)
        if dims:
            return dims
    return None


def _deflate_volume(L_box_axes, V_box_scalar, orig_clear, new_clear):
    """
    Recompute V_mix for a new clearance.
    Occupied extent per axis = L_box - 2*orig_clear; new box adds 2*new_clear.
    If per-axis dims are unknown, fall back to an isotropic (cube-root) estimate,
    which UNDER-counts the effect for a slab (thin axis loses a larger fraction).
    """
    delta = 2.0 * (orig_clear - new_clear)   # subtract this length from each axis
    if L_box_axes is not None:
        return float(np.prod([L - delta for L in L_box_axes])), "exact"
    L = V_box_scalar ** (1.0 / 3.0)
    return float((L - delta) ** 3), "approx(cube)"


rows = []
methods_used = set()

for P in PRESSURES:
    pstr = f"{P:.1f}"
    dataname          = f"{BASE_DATANAME}_pstar{pstr}"
    isolated_dataname = f"isolated_{dataname}_{INTERACTION}_{SLAB_STEPS}"
    sol_dataname      = f"{isolated_dataname}_solvent_only"
    pol_dataname      = f"{isolated_dataname}_polymer_only"
    p_dir = DATA_DIR / f"p{pstr}"

    manifest_file = p_dir / f"{isolated_dataname}_volmix_manifest.json"
    sol_vol_file  = p_dir / f"box_dimensions_{sol_dataname}_{PURE_INTER}_{PURE_STEPS}.dat"
    pol_vol_file  = p_dir / f"box_dimensions_{pol_dataname}_{PURE_INTER}_{PURE_STEPS}.dat"
    if not all(f.exists() for f in [manifest_file, sol_vol_file, pol_vol_file]):
        print(f"[SKIP] P*={pstr}: missing manifest/solvent/polymer file")
        continue

    mf        = load_manifest(manifest_file)
    V_box0    = mf["V_mix_isolated"]                       # geometric box at clearance 0.2
    scale_pol = mf["scale_pol"]
    scale_sol = mf["scale_sol"]
    V_pol_ref = avg_box_volume(pol_vol_file, skip_frac=0.5) * scale_pol
    V_sol_ref = avg_box_volume(sol_vol_file, skip_frac=0.5) * scale_sol
    V_ref     = V_pol_ref + V_sol_ref

    L_axes = _find_isolated_dims(p_dir, isolated_dataname)

    row = {"P": P, "V_ref": V_ref}
    for c in TEST_CLEARANCES:
        V_mix_c, method = _deflate_volume(L_axes, V_box0, ORIG_CLEARANCE, c)
        methods_used.add(method)
        row[f"c{c}"] = (V_mix_c - V_ref) / V_ref       # fractional ΔV_mix
    rows.append(row)

if not rows:
    print("No data — check that the staged manifest/box_dimensions files exist.")
else:
    P_arr = np.array([r["P"] for r in rows])

    # --- Plot: ΔV_mix/(V_sol+V_pol) vs P* for each clearance ---
    fig, ax = plt.subplots(figsize=(7, 5))
    print(f"\n{'P*':>5}", *[f"c={c:<5}" for c in TEST_CLEARANCES], sep="  ")
    for c in TEST_CLEARANCES:
        y = np.array([r[f"c{c}"] for r in rows])
        slope = np.polyfit(P_arr, y, 1)[0]
        ax.plot(P_arr, y, "o-", lw=1.8, ms=6,
                label=f"clearance={c}  (slope={slope:+.4f}/P*)")
    ax.axhline(0, color="gray", lw=1, ls="--")
    ax.set_xlabel(r"$P^*$")
    ax.set_ylabel(r"$\Delta V_{\rm mix}\,/\,(V_{\rm sol}+V_{\rm pol})$")
    ax.set_title("Clearance sensitivity of ΔV_mix")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # --- Table ---
    print()
    for r in rows:
        vals = "  ".join(f"{r[f'c{c}']:+.4f}" for c in TEST_CLEARANCES)
        print(f"{r['P']:>5.1f}  {vals}")

    print(f"\nDeflation method(s) used: {sorted(methods_used)}")
    if "approx(cube)" in methods_used:
        print("NOTE: cube-root fallback under-counts the clearance effect for a "
              "slab geometry.\n      For exact per-axis deflation, make the "
              "isolated_*.data files reachable\n      (edit the candidate paths in "
              "_find_isolated_dims, or stage them from Expanse).")
    print("\nRead-out: if the slope shrinks toward ~0 as clearance → 0, the upward "
          "trend\nwas the fixed-clearance artifact. If it survives, it's the real "
          "packing effect.")